In [5]:
import datasets
import sys
import os
from evaluate import load as load_metric

sys.path.insert(0, "generation_code")
from utils import get_regesto_prompt, join_lines
from generate_ita_regesti import prepare_inputs, generate
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

from vllm import LLM, SamplingParams


In [6]:
dss = [datasets.load_dataset("json", data_files=f"output/escriptorium_mgh_{i}.json", split="train") for i in range(1,4)]
ds = datasets.concatenate_datasets(dss)
ds = ds.train_test_split(test_size=0.1)


In [7]:
def get_model_data_and_compute(model_name, ds):
    llm = LLM(model_name)
    sampling_params = SamplingParams(
        temperature=0,
        top_p=0.95,
        top_k=40,
        max_tokens=512,
    )

    bleu = load_metric("bleu")
    rouge = load_metric("rouge")
    full_input = ds["train"]
    inputs = [get_regesto_prompt(join_lines(i) if i is not None else "", [], [], dataset_name="mgh", n=0) for i in full_input["testo esteso"]]
    prepared_inputs = prepare_inputs(inputs, llm)
    output = llm.generate(
        prepared_inputs,
        sampling_params=sampling_params,
    )

    for i, j in zip(output, full_input["regesto"]):
        if j is None:
            continue
        rouge.add(
            predictions=i.outputs[0].text,
            reference=join_lines(j)
        )
        bleu.add(
            predictions=i.outputs[0].text,
            reference=join_lines(j)
        )

    return {"rouge": rouge.compute(), "bleu": bleu.compute()}, output


In [8]:
# scores_llama_finetune, output_llama_finetune = get_model_data_and_compute("./llama_3.1_8b_instruct_Regesta_SFT_bs4_full_model/", ds)
scores_llama_finetune, output_llama_finetune = get_model_data_and_compute("./llama_3.1_8b_instruct_Regesta_SFT_bs4_5_epochs_full_model", ds)
# output_llama_instruct = get_model_data_and_compute("meta-llama/Llama-3.1-8b-Instruct", ds)

INFO 05-19 18:37:01 [config.py:2968] Downcasting torch.float32 to torch.float16.
INFO 05-19 18:37:01 [config.py:717] This model supports multiple tasks: {'score', 'reward', 'classify', 'embed', 'generate'}. Defaulting to 'generate'.
INFO 05-19 18:37:03 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 05-19 18:37:05 [core.py:58] Initializing a V1 LLM engine (v0.8.5.post1) with config: model='./llama_3.1_8b_instruct_Regesta_SFT_bs4_5_epochs_full_model', speculative_config=None, tokenizer='./llama_3.1_8b_instruct_Regesta_SFT_bs4_5_epochs_full_model', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=131072, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config

Loading safetensors checkpoint shards:   0% Completed | 0/7 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  14% Completed | 1/7 [00:01<00:08,  1.45s/it]
Loading safetensors checkpoint shards:  29% Completed | 2/7 [00:03<00:08,  1.70s/it]
Loading safetensors checkpoint shards:  43% Completed | 3/7 [00:05<00:08,  2.09s/it]
Loading safetensors checkpoint shards:  57% Completed | 4/7 [00:07<00:05,  1.72s/it]
Loading safetensors checkpoint shards:  71% Completed | 5/7 [00:10<00:04,  2.21s/it]
Loading safetensors checkpoint shards:  86% Completed | 6/7 [00:13<00:02,  2.56s/it]
Loading safetensors checkpoint shards: 100% Completed | 7/7 [00:16<00:00,  2.62s/it]
Loading safetensors checkpoint shards: 100% Completed | 7/7 [00:16<00:00,  2.30s/it]



INFO 05-19 18:37:27 [loader.py:458] Loading weights took 16.73 seconds
INFO 05-19 18:37:28 [gpu_model_runner.py:1347] Model loading took 14.9889 GiB and 17.125374 seconds
INFO 05-19 18:37:44 [backends.py:420] Using cache directory: /raid/homes/giovanni.puccetti/.cache/vllm/torch_compile_cache/c29d63dd21/rank_0_0 for vLLM's torch.compile
INFO 05-19 18:37:44 [backends.py:430] Dynamo bytecode transform time: 15.60 s
INFO 05-19 18:37:55 [backends.py:118] Directly load the compiled graph(s) for shape None from the cache, took 9.911 s
INFO 05-19 18:37:59 [monitor.py:33] torch.compile takes 15.60 s in total
INFO 05-19 18:38:01 [kv_cache_utils.py:634] GPU KV cache size: 194,592 tokens
INFO 05-19 18:38:01 [kv_cache_utils.py:637] Maximum concurrency for 131,072 tokens per request: 1.48x


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f71630b7c20>>
Traceback (most recent call last):
  File "/raid/homes/giovanni.puccetti/Repos/MGH_annotation/conda_venv/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

  File "/raid/homes/giovanni.puccetti/Repos/MGH_annotation/conda_venv/lib/python3.12/site-packages/vllm/v1/engine/core.py", line 371, in signal_handler
    raise SystemExit()
SystemExit: 


KeyboardInterrupt: 

In [ ]:
print("LLAMA FINETUNES:")
print(scores_llama_finetune)
# print()
# print("-"*50)
# print()
# print("LLAMA INSTRUCT:")
# print(output_llama_instruct)

LLAMA FINETUNES:
{'rouge': {'rouge1': np.float64(0.15152301083171296), 'rouge2': np.float64(0.0700360107562861), 'rougeL': np.float64(0.13615720832316375), 'rougeLsum': np.float64(0.1360955067780926)}, 'bleu': {'bleu': 0.031498551947862594, 'precisions': [0.08344519433267772, 0.03921010999266034, 0.023123234484396814, 0.013011110977728313], 'brevity_penalty': 1.0, 'length_ratio': 5.950329186525567, 'translation_length': 589273, 'reference_length': 99032}}


In [ ]:
from pprint import pprint
idx = 1
pprint(" ".join(ds["train"]["testo esteso"][idx])[:300])
pprint(" ".join(ds["train"]["regesto"][idx]))
pprint(output_llama_finetune[idx].outputs[0].text, )

('Iohanni de Civitella subdiacono et notariot, capellano nostro. Cum, sicut ex '
 'parte venerabilis fratris nostri .. Vesprimiensis episcopi nobis extitit '
 'intimatum, tam gravi corporis infirmitate laboret, quod personaliter ad con¬ '
 'cilium, quod Domino prosperante celebrare proponimus, venire non potest, ')
('Gregorius IX papa Iohanni de Civitella subdiacono mandat, ut si (Briccius) '
 'epi¬ scopus Vesprimiensis propter corporis infirmitatem ad concilium venire '
 'nequeat, ei rema¬ nendi licentiam tribuat. 1241, Mart. 18.')
('Honorius III papa Iohanni de Civitella subdiacono et notario capellano suo '
 'mandat, ut (Willelmo) Vesprimiensi episcopo, si ad concilium celebrandum '
 'personaliter venire non possit, auctoritate sua licentiam remanendi tribuat. '
 '1220, Mart. 18. Innocentius IV Pontificis regesto non habet. Cf. n. 104. 1. '
 '1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. '
 '1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 